# 🧬 TDC-Studio SOTA ADME Training Runner (Google Colab)

이 노트북은 **Google Colab 클라우드 환경(GPU/CPU)**에서 TDC-Studio의 ADME 벤치마크 SOTA 파이프라인을 실행합니다.

### 실행 대상 파이프라인:
1. **간세포 클리어런스 (`Clearance_Hepatocyte_AZ`)**: RDKit 200+ Descriptors + Multi-Task D-MPNN 결측 없는 SOTA Prior + MAE/Huber 순위 보존 특화 앙상블 (목표: Spearman $\rho \ge 0.62$)
2. **지질친화도 (`Lipophilicity_AstraZeneca`)**: 24-dim 물화 모티프 + ChemBERTa 언어 임베딩 + 4-Layer D-MPNN 위상 표현 Tri-Hybrid 스태킹 (목표: $R^2 \ge 0.85$)

Bemis-Murcko Scaffold Split (`seed=42`) 엄격 기준 평가.

In [ ]:
# 1. GPU 하드웨어 가속기 상태 점검
!nvidia-smi


In [ ]:
# 2. 초고속 패키지 관리자 uv 설치 및 PATH 설정
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ['PATH']
!uv --version


In [ ]:
# 3. TDC-Studio 최신 원격 저장소 복제 및 가상환경 동기화
import os
if not os.path.exists('tdc-studio'):
    !git clone https://github.com/brightonmoon/tdc-studio.git
%cd tdc-studio
!git pull origin main

# 의존성 설치 (RDKit, PyG, CatBoost, Transformers 등)
!uv sync --extra tdc


In [ ]:
# 4. W&B Model Registry에서 기학습된 Multi-Task D-MPNN 체크포인트 동기화 (옵션)
!uv run python scripts/sync_wandb_models.py || true


In [ ]:
# 5. [핵심] 간세포 클리어런스 Rank-Aware Multi-Objective SOTA 학습 실행
# - RDKit 200+ 전체 기술자
# - D-MPNN 마이크로솜/반감기/PPBR 연속 Prior
# - MAE (55%) + Huber (45%) 3-시드 앙상블
!uv run python deploy/train_clearance_cascade.py


In [ ]:
# 6. [선택] 지질친화도 Tri-Hybrid D-MPNN 스태킹 학습 실행
# - 24 물화 모티프 + ChemBERTa-77M + 4-Layer D-MPNN
!uv run python deploy/train_lipo_stacking.py


In [ ]:
# 7. 학습 결과 요약 JSON 확인
import json
from pathlib import Path

for summary_file in Path('models/export').glob('*/*summary*.json'):
    print(f"\n=== {summary_file} ===")
    print(json.dumps(json.loads(summary_file.read_text(encoding='utf-8')), indent=2))
